## Cargar variables de entorno

In [1]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv(dotenv_path=".env", override=True)

True

## Crear aplicación AI 

### Setup 

Como siempre, definamos nuestro prompt y demos a nuestra aplicación acceso a la web.

In [2]:
# Inicializar herramienta de búsqueda web.
from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=1)

# Definir prompt template
prompt = """Sos un profesor y un experto en explicar temas complejos de una manera fácil de entender.
Tu trabajo es responder la pregunta dada de forma que incluso un niño de 5 años pueda comprenderla.
Se te ha brindado el contexto necesario para responder la pregunta.

Pregunta: {question} 

Contexto: {context}

Respuesta:"""

C:\Users\sergi\AppData\Local\Temp\ipykernel_14752\750042844.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\sergi\AppData\Local\Temp\ipykernel_14752\750042844.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search_tool = TavilySearchResults(max_results=1)


### Definir la lógica de la aplicación.

La lógica acá es la misma que en el módulo de trazas. Definimos un paso de búsqueda para explorar la web y un paso de explicación para que un modelo de lenguaje resuma los resultados encontrados.

In [3]:
from openai import OpenAI
from langsmith import traceable
from langsmith.wrappers import wrap_openai


# Crear application
openai_client = wrap_openai(OpenAI())

@traceable
def search(question):
    web_docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in web_docs])
    return web_results
    
@traceable
def explain(question, context):
    formatted = prompt.format(question=question, context=context)
    
    completion = openai_client.chat.completions.create(
        messages=[
            {"role": "system", "content": formatted},
            {"role": "user", "content": question},
        ],
        model="o3-mini",
    )
    return completion.choices[0].message.content

@traceable
def eli5(question):
    context = search(question)
    answer = explain(question, context)
    return answer


## Setup del experimento

Ahora estamos listos para ejecutar experimentos y probar el rendimiento de nuestra aplicación sobre nuestro dataset.

### Importar cliente LangSmith 

Primero, vamos a crear un cliente de LangSmith para usar el SDK y especificar el dataset sobre el que queremos ejecutar nuestro experimento.

In [4]:
from langsmith import Client

client = Client()
#dataset_name = "eli5-silver"
dataset_name = "ds-silver-turmeric-30"

### Definir evaluadores

#### Evaluador de código personalizado

Primero definiremos un evaluador de código personalizado, que resulta útil para medir métricas deterministas o de respuesta cerrada.

In [5]:
def conciseness(outputs: dict) -> bool:
    words = outputs["output"].split(" ")
    return len(words) <= 200

Este evaluador de código personalizado es simplemente una función de Python que verifica si nuestra aplicación produce respuestas de 200 palabras o menos.

#### LLM-as-a-Judge Evaluador

Para métricas abiertas, puede ser muy potente usar un LLM para puntuar las respuestas.

Usemos un LLM para comprobar si nuestra aplicación produce resultados correctos. Primero, definamos un esquema de puntuación que nuestro LLM deba seguir en su respuesta.

In [6]:
from pydantic import BaseModel, Field

# Definir un esquema de puntuación al que nuestro LLM debe ajustarse.
class CorrectnessScore(BaseModel):
    """Correctness score of the answer when compared to the reference answer."""
    score: int = Field(description="The score of the correctness of the answer, from 0 to 1")

Vamos a definir una función para darle a un LLM las salidas de nuestra aplicación, junto con las salidas de referencia guardadas en nuestro conjunto de datos.

De este modo, el LLM podrá usar la respuesta “correcta” como referencia para juzgar si la respuesta de nuestra aplicación cumple con nuestros estándares de precisión.

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    prompt = """
    Sos un etiquetador de datos experto que evalúa las respuestas de un modelo para verificar su corrección.
Tu tarea es asignar una puntuación basada en la siguiente rúbrica:

    <Rubric>
        Una respuesta correcta:
            - Brinda información precisa
            - Usa analogías y ejemplos adecuados
            - No contiene errores fácticos
            - Es lógicamente consistente

        Al puntuar, debés penalizar:
            - Errores fácticos
            - Analogías y ejemplos incoherentes
            - Inconsistencias lógicas    
    </Rubric>

    <Instructions>
        - Leé con atención la entrada y la salida.
        - Usá la salida de referencia para determinar si la salida del modelo contiene errores.
        - Concentrate en si la salida del modelo usa analogías precisas y es lógicamente consistente.
    </Instructions>

    <Reminder>
        Las analogías de la salida no necesitan coincidir exactamente con la salida de referencia. Concentrate en la consistencia lógica.
    </Reminder>

    <input>
        {}
    </input>

    <output>
        {}
    </output>

    Usá las salidas de referencia que aparecen abajo para ayudarte a evaluar la corrección de la respuesta.
    <reference_outputs>
        {}
    </reference_outputs>
    """.format(inputs["question"], outputs["output"], reference_outputs["output"])
    structured_llm = ChatOpenAI(model_name="gpt-4o", temperature=0).with_structured_output(CorrectnessScore)
    generation = structured_llm.invoke([HumanMessage(content=prompt)])
    return generation.score == 1


### Definir Run Function

Vamos a definir una función para ejecutar nuestra aplicación sobre las entradas de ejemplo de nuestro dataset. Esta es la función que se va a llamar cuando ejecutemos nuestro experimento.

In [8]:
# Definir una función para ejecutar tu aplicación.
def run(inputs: dict):
    return eli5(inputs["question"])

## Correr experimento

Tenemos todos los componentes necesarios, así que ejecutemos nuestro experimento!

In [9]:
from langsmith import evaluate

evaluate(
    run,
    data=dataset_name,
    evaluators=[correctness, conciseness],
    experiment_prefix="eli5-o3-mini"
)

View the evaluation results for experiment: 'eli5-o3-mini-523cb227' at:
https://smith.langchain.com/o/11afda74-8804-4cb8-8ad4-c2a9d67d44e5/datasets/1f1c61b4-2861-4ff1-a1f0-5e67d5d7fad5/compare?selectedSessions=2f2df04a-7260-4e91-bf26-f8055f36236c




c:\Users\sergi\OneDrive\Documentos\UCEMA\cursoGenAI-LangSmith\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
10it [01:17,  7.80s/it]


,inputs.question,outputs.output,error,reference.output,feedback.correctness,feedback.conciseness,execution_time,example_id,id
0,Qué es la macroeconomía?,La macroeconomía es como mirar la gran imagen ...,None,La macroeconomía se ocupa de entender el funci...,False,True,6.479126,31fce7e9-b492-4f1e-b965-edff69da8ffb,019eb8d4-4a8b-7fa3-8948-0ed3d4dd962d
1,How does photosynthesis work?,Imagine that plants have tiny kitchens inside ...,None,"Okay! Imagine plants are like tiny chefs, and ...",True,True,7.934241,483c86eb-edad-47a8-b415-6c72c889b39d,019eb8d4-6d1f-70a3-9e7d-f9045d2b7a6c
2,How does string theory work?,Imagina que todo en el universo está hecho de ...,None,"Okay! Imagine that everything in the universe,...",True,False,7.935745,5e92d0ac-980c-47dd-9102-795e59cb9557,019eb8d4-8f64-7ac2-b621-0122a4bc0330
3,How does a democracy work?,Imagine that a democracy is like a big classro...,None,Okay! Imagine you and your friends want to dec...,True,True,6.902259,65a532c5-444e-4b68-b1d5-d983770353bd,019eb8d4-b17f-7b62-a045-b30f197a05ea
4,Why is the sky blue?,Imagine that sunlight is like a box of crayons...,None,Alright! Imagine the sky is like a big bowl of...,True,True,7.690537,670f453e-7960-47c5-a99e-9f721f843beb,019eb8d4-d055-7ae2-b845-a24de6750db6
5,What is LangGraph?,Imagínate que tienes una caja llena de piezas ...,None,"Okay, imagine you have a big box of LEGO brick...",True,True,7.265447,916d89c2-322e-4643-9738-16d350dcd281,019eb8d4-f19f-71c2-a949-ba3312e5c45a
6,What is trustcall library?,Imagine you have a big box of LEGO bricks that...,None,"Alright, imagine you have a toy box where each...",False,True,5.717675,ab7119f1-c555-4ab5-8c1a-f93a8dce17a4,019eb8d5-12c9-7d03-834e-343a460fa439
7,What is the Langchain framework?,Imagina que tienes un enorme juguete de bloque...,None,Okay! Imagine you want to build a really cool ...,True,True,5.281298,cc8cd873-1132-417d-80fb-067b10315de0,019eb8d5-2cf0-7661-ba68-300ccd773c74
8,What is sound?,Imagina que tienes un tambor. Cuando lo golpea...,None,Okay! Imagine you have a drum. When you hit it...,True,True,5.978129,f76532c5-d461-4cdf-a2c2-a49f671f22b7,019eb8d5-452e-7532-b6b5-985df3d38a7e
9,What is LangSmith by LangChain?,Imagina que estás construyendo una gran torre ...,None,Okay! Imagine you have a big box of toys that ...,True,True,5.689402,ffdf4299-3ca1-428a-a931-6244d172feb1,019eb8d5-608c-7c81-affb-99ca9e802e7a
